# Patrón Creacional: Singleton — Cafetería

**Dominio propio:** configuración global de la cafetería (impuesto, moneda, nombre de la sucursal) compartida por toda la app.

## Introducción — qué problema resuelve
En la app de la cafetería, varios módulos (la caja, el generador de recibos, el panel de reportes) necesitan leer la **misma** configuración: el porcentaje de impuesto, la moneda y el nombre de la sucursal. Si cada módulo crea su propio objeto de configuración, terminan con **copias desincronizadas**: cambio el impuesto en un lado y el otro sigue usando el valor viejo.

El patrón **Singleton** garantiza una única instancia y un punto de acceso global, así toda la app comparte exactamente la misma configuración.

## Sin patrón (el problema es evidente)

Cada módulo instancia su propia `ConfiguracionCafeteria`. Cuando la caja actualiza el impuesto, el módulo de recibos no se entera: son objetos distintos.

In [1]:
class ConfiguracionCafeteria:
    def __init__(self) -> None:
        self.sucursal: str = "Centro"
        self.moneda: str = "COP"
        self.impuesto: float = 0.08


# El modulo de caja crea su propia configuracion...
config_caja = ConfiguracionCafeteria()
config_caja.impuesto = 0.19  # sube el IVA

# ...y el modulo de recibos crea OTRA distinta
config_recibos = ConfiguracionCafeteria()

print("Impuesto en caja:   ", config_caja.impuesto)
print("Impuesto en recibos:", config_recibos.impuesto)  # sigue en 0.08, desincronizado
print("¿Misma instancia?   ", config_caja is config_recibos)

Impuesto en caja:    0.19
Impuesto en recibos: 0.08
¿Misma instancia?    False


### Problema visible
El impuesto se actualizó en la caja (`0.19`) pero el módulo de recibos sigue calculando con `0.08`. Son dos objetos separados: no hay una única fuente de verdad.

## Con patrón Singleton (problema resuelto)

Sobrescribo `__new__` para que la clase devuelva **siempre la misma instancia**. No importa cuántos módulos pidan la configuración: todos comparten el mismo objeto.

In [2]:
class ConfiguracionCafeteria:
    _instancia: 'ConfiguracionCafeteria | None' = None

    def __new__(cls) -> 'ConfiguracionCafeteria':
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            # Inicializacion unica de los valores por defecto
            cls._instancia.sucursal = "Centro"
            cls._instancia.moneda = "COP"
            cls._instancia.impuesto = 0.08
        return cls._instancia

    def precio_con_impuesto(self, precio: float) -> float:
        return precio * (1 + self.impuesto)


# Todos los modulos obtienen la MISMA instancia
config_caja = ConfiguracionCafeteria()
config_caja.impuesto = 0.19  # la caja sube el IVA

config_recibos = ConfiguracionCafeteria()  # el modulo de recibos pide la config

print("Impuesto en caja:   ", config_caja.impuesto)
print("Impuesto en recibos:", config_recibos.impuesto)  # 0.19, sincronizado
print("¿Misma instancia?   ", config_caja is config_recibos)
print("Precio de $10000 con impuesto:", config_recibos.precio_con_impuesto(10000))

Impuesto en caja:    0.19
Impuesto en recibos: 0.19
¿Misma instancia?    True
Precio de $10000 con impuesto: 11900.0


### Resultado
Ahora `config_caja` y `config_recibos` son **el mismo objeto** (`is` da `True`). El cambio de impuesto se refleja en toda la app: una única fuente de verdad.

## Diagrama UML (clases reales de este ejemplo)
```plantuml
@startuml
class ConfiguracionCafeteria {
    - _instancia: ConfiguracionCafeteria
    + sucursal: str
    + moneda: str
    + impuesto: float
    + __new__(): ConfiguracionCafeteria
    + precio_con_impuesto(precio): float
}
ConfiguracionCafeteria ..> ConfiguracionCafeteria : devuelve la misma _instancia
@enduml
```

## ¿Por qué Singleton y no otro patrón creacional?

El problema aquí no es **cómo construir** objetos variados (eso sería Factory Method o Abstract Factory), ni ensamblar un objeto complejo paso a paso (Builder), ni clonar objetos existentes (Prototype). El problema es exactamente el contrario: quiero **evitar** que existan múltiples objetos y garantizar que solo haya **uno** compartido por toda la app.

Como la configuración es un recurso global que debe ser único y consistente (igual que un logger o un pool de conexiones), **Singleton** es el patrón creacional que ataca justo ese problema: instancia única + punto de acceso global.